## Precursor-level quantification

After XIC-level quantification, the next step is to summrize the XIC quantities to the precursor-level. The MS1- and MS2-level XIC quantities were summarized into MS1- and MS2-level precursor quantities, respectively.

The Precursor-XIC quantity mapping data is stored in the `intermediate/prec_quant` folder in the output directory.

### Generate MS2-level quantities

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import polars as pl

prec_id = 'INEKPTVVNDYEAAR2'
quantity_df = (
    pl.scan_parquet("../tests/toy_Precursor.Id_MS2.XIC.Quant.Raw_mapping.parquet")
    .filter(pl.col("target_id") == prec_id)
    .collect()
)
quantity_df.head(5)

target_id,samples,source_id,source_quantity
str,str,str,f64
"""INEKPTVVNDYEAAR2""","""20230222_Exp_SM_MethMS_manual_…","""y11^1/1234.6062""",397152.582031
"""INEKPTVVNDYEAAR2""","""20230222_Exp_SM_MethMS_manual_…","""b3^1/357.1768""",108312.434814
"""INEKPTVVNDYEAAR2""","""20230222_Exp_SM_MethMS_manual_…","""y8^1/937.4374""",42411.6875
"""INEKPTVVNDYEAAR2""","""20230222_Exp_SM_MethMS_manual_…","""y10^1/1137.5535""",17751.341797
"""INEKPTVVNDYEAAR2""","""20230222_Exp_SM_MethMS_manual_…","""y11^2/617.8068""",10197.624023


In [2]:
quality_df = (
    pl.scan_parquet("../tests/toy_Precursor.Id_MS2.XIC.Corr.Raw_mapping.parquet")
    .filter(pl.col("target_id") == prec_id)
    .collect()
)
quality_df.head(5)

target_id,samples,source_id,source_quality
str,str,str,f64
"""INEKPTVVNDYEAAR2""","""20230222_Exp_SM_MethMS_manual_…","""y11^1/1234.6062""",0.871727
"""INEKPTVVNDYEAAR2""","""20230222_Exp_SM_MethMS_manual_…","""b3^1/357.1768""",0.876633
"""INEKPTVVNDYEAAR2""","""20230222_Exp_SM_MethMS_manual_…","""y8^1/937.4374""",0.884939
"""INEKPTVVNDYEAAR2""","""20230222_Exp_SM_MethMS_manual_…","""y10^1/1137.5535""",0.761785
"""INEKPTVVNDYEAAR2""","""20230222_Exp_SM_MethMS_manual_…","""y11^2/617.8068""",0.761785


#### 1. Generate quantity and quality matrices

In [3]:
# Pivot input tables
quantity_pivoted = quantity_df.pivot(
    values="source_quantity",
    index=["target_id", "source_id"],
    on="samples",
    aggregate_function="first"
)
quality_pivoted = quality_df.pivot(
    values="source_quality",
    index=["target_id", "source_id"],
    on="samples",
    aggregate_function="first"
)

sample_cols = [col for col in quantity_pivoted.columns if col not in {"target_id", "source_id"}]

quantity_df_sub = quantity_pivoted
quality_df_sub = quality_pivoted

quantity_mat = quantity_df_sub.select(sample_cols).to_numpy()
quality_mat = quality_df_sub.select(sample_cols).to_numpy()

print(f"MS2-level quantity matrix for {prec_id}")
pd.DataFrame(np.log2(quantity_mat+1))

MS2-level quantity matrix for INEKPTVVNDYEAAR2


,0,1,2,3,4,5,6,7
0,18.599337,18.330869,18.452302,18.146264,17.059033,17.046791,17.308915,16.876468
1,16.724853,16.547050,16.628020,16.087524,15.070452,15.276017,15.676706,14.584805
2,15.372208,15.233611,15.182970,14.682457,13.888845,13.460569,13.956219,13.589827
3,14.115722,13.660493,13.783531,13.876178,12.801294,12.088125,13.628417,12.808693
4,13.316087,13.491762,13.109120,13.047645,10.964959,11.524034,12.813047,11.332840
5,12.737116,11.896661,13.033446,12.720181,11.568436,0.000000,0.000000,0.000000


In [4]:
print(f"MS2-level quality matrix for {prec_id}")
pd.DataFrame(quality_mat)

MS2-level quality matrix for INEKPTVVNDYEAAR2


,0,1,2,3,4,5,6,7
0,0.871727,0.879824,0.869639,0.918123,0.876735,0.868252,0.938476,0.948605
1,0.876633,0.877182,0.883267,0.943297,0.897558,0.810145,0.961867,0.881018
2,0.884939,0.890812,0.859476,0.922732,0.870828,0.764419,0.911625,0.934089
3,0.761785,0.754313,0.850950,0.867570,0.615440,0.764419,0.762159,0.913142
4,0.761785,0.754313,0.756453,0.695783,0.747397,0.764419,0.926358,0.645212
5,0.761785,0.754313,0.756453,0.930109,0.747397,0.000000,0.000000,0.000000


#### 2. Filtering of low quality ions

In [5]:
import numpy as np

use_lion_inference = True
xic_quality_cutoff = 0.0
allowed_xic_missing = 0.5

quantity_mat_fill = quantity_mat.copy()
if use_lion_inference:
    if xic_quality_cutoff <= 0.0:
        missing_mask = (quality_mat < xic_quality_cutoff) | (quality_mat == 0.0)
    else:
        missing_mask = quality_mat < xic_quality_cutoff
    missing_mask = missing_mask | (quantity_mat == 0.0)
    quantity_mat_fill[missing_mask] = np.nan
    # remove rows with x% or more NaNs
    row_nan_percentage = np.isnan(quantity_mat_fill).sum(axis=1) / quantity_mat_fill.shape[1]
    quantity_mat_fill = quantity_mat_fill[row_nan_percentage < allowed_xic_missing]

pd.DataFrame(np.log2(quantity_mat_fill+1))

,0,1,2,3,4,5,6,7
0,18.599337,18.330869,18.452302,18.146264,17.059033,17.046791,17.308915,16.876468
1,16.724853,16.547050,16.628020,16.087524,15.070452,15.276017,15.676706,14.584805
2,15.372208,15.233611,15.182970,14.682457,13.888845,13.460569,13.956219,13.589827
3,14.115722,13.660493,13.783531,13.876178,12.801294,12.088125,13.628417,12.808693
4,13.316087,13.491762,13.109120,13.047645,10.964959,11.524034,12.813047,11.332840
5,12.737116,11.896661,13.033446,12.720181,11.568436,NaN,NaN,NaN


#### 3. Perform LION inference

In [6]:
import pandas as pd
from jumplion.utils import perform_lion_inference

sample_meta = pd.read_csv("../tests/sample_meta.csv")
min_n_xic = 2

# perform LION inference
if quantity_mat_fill.shape[0] < min_n_xic:
    pass
else:
    quantity_mat_fill = perform_lion_inference(quantity_mat_fill, 
                                               sample_cols, 
                                               sample_meta,
                                               sample_column='Sample', 
                                               group_column='Group',
                                               lion_value_type='local_min',
                                               lion_value_scale=1.0)
pd.DataFrame(np.log2(quantity_mat_fill))

,0,1,2,3,4,5,6,7
0,18.599334,18.330864,18.452298,18.146259,17.059023,17.046780,17.308906,16.876456
1,16.724839,16.547035,16.628005,16.087503,15.070410,15.275981,15.676678,14.584746
2,15.372174,15.233573,15.182931,14.682402,13.888750,13.460441,13.956128,13.589710
3,14.115640,13.660382,13.783429,13.876082,12.801091,12.087793,13.628303,12.808492
4,13.315945,13.491637,13.108957,13.047475,10.964237,11.523544,12.812846,11.332281
5,12.736904,11.896283,13.033274,12.719968,11.567961,10.964237,10.964237,10.964237


#### 4. Summarization

In [7]:
from jumplion.summarization import median_summ

quantity_lion = median_summ(quantity_mat_fill, topN_rep=3, rep_abundance='mean')
np.log2(quantity_lion)

array([17.47147877, 17.25805345, 17.33853607, 17.06989825, 15.98878458,
       15.68618753, 16.31189581, 15.72370603])

### Generate MS1-level quantities

#### 1. Generate quantity and quality matrices

In [8]:
quantity_df = (
    pl.scan_parquet("../tests/toy_Precursor.Id_MS1.XIC.Quant.Adjusted_mapping.parquet")
    .filter(pl.col("target_id") == prec_id)
    .collect()
)
quality_df = (
    pl.scan_parquet("../tests/toy_Precursor.Id_MS1.XIC.Corr.Raw_mapping.parquet")
    .filter(pl.col("target_id") == prec_id)
    .collect()
)

# Pivot input tables
quantity_pivoted = quantity_df.pivot(
    values="source_quantity",
    index=["target_id", "source_id"],
    on="samples",
    aggregate_function="first"
)
quality_pivoted = quality_df.pivot(
    values="source_quality",
    index=["target_id", "source_id"],
    on="samples",
    aggregate_function="first"
)

sample_cols = [col for col in quantity_pivoted.columns if col not in {"target_id", "source_id"}]

quantity_df_sub = quantity_pivoted
quality_df_sub = quality_pivoted

quantity_mat = quantity_df_sub.select(sample_cols).to_numpy()
quality_mat = quality_df_sub.select(sample_cols).to_numpy()

print(f"MS1-level quantity matrix for {prec_id}")
pd.DataFrame(quantity_mat)

MS1-level quantity matrix for INEKPTVVNDYEAAR2


,0,1,2,3,4,5,6,7
0,350395.116817,322093.959561,339091.860523,243896.736777,112124.128531,89596.982075,84807.943978,68842.056727
1,338656.012618,309862.716840,253225.594554,229272.348211,74119.107786,0.000000,73198.336020,131310.929113
2,16441.988966,150165.050075,103156.233155,93242.100806,0.000000,0.000000,52134.928619,0.000000
3,10106.382924,0.000000,0.000000,57642.921197,44532.646312,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [9]:
print(f"MS1-level quality matrix for {prec_id}")
pd.DataFrame(quality_mat)

MS1-level quality matrix for INEKPTVVNDYEAAR2


,0,1,2,3,4,5,6,7
0,0.990721,0.957188,0.984245,0.955627,0.910639,0.990615,0.87356,0.890485
1,0.948414,0.979251,0.977212,0.940372,0.910639,0.000000,0.87356,0.966584
2,0.689645,0.934004,0.902102,0.925527,0.000000,0.000000,0.87356,0.000000
3,0.689645,0.000000,0.000000,0.925527,0.910639,0.000000,0.00000,0.000000
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000


#### 2. Filtering of low quality ions

In [10]:
use_lion_inference = True
xic_quality_cutoff = 0.0
allowed_xic_missing = 0.5

quantity_mat_fill = quantity_mat.copy()
if use_lion_inference:
    if xic_quality_cutoff <= 0.0:
        missing_mask = (quality_mat < xic_quality_cutoff) | (quality_mat == 0.0)
    else:
        missing_mask = quality_mat < xic_quality_cutoff
    missing_mask = missing_mask | (quantity_mat == 0.0)
    quantity_mat_fill[missing_mask] = np.nan
    # remove rows with x% or more NaNs
    row_nan_percentage = np.isnan(quantity_mat_fill).sum(axis=1) / quantity_mat_fill.shape[1]
    quantity_mat_fill = quantity_mat_fill[row_nan_percentage < allowed_xic_missing]

pd.DataFrame(quantity_mat_fill)

,0,1,2,3,4,5,6,7
0,350395.116817,322093.959561,339091.860523,243896.736777,112124.128531,89596.982075,84807.943978,68842.056727
1,338656.012618,309862.716840,253225.594554,229272.348211,74119.107786,NaN,73198.336020,131310.929113
2,16441.988966,150165.050075,103156.233155,93242.100806,NaN,NaN,52134.928619,NaN


#### 3. Perform LION inference

In [11]:
# perform LION inference
if quantity_mat_fill.shape[0] < min_n_xic:
    pass
else:
    quantity_mat_fill = perform_lion_inference(quantity_mat_fill, 
                                               sample_cols, 
                                               sample_meta,
                                               sample_column='Sample', 
                                               group_column='Group',
                                               lion_value_type='local_min',
                                               lion_value_scale=1.0)
pd.DataFrame(quantity_mat_fill)

,0,1,2,3,4,5,6,7
0,350395.116817,322093.959561,339091.860523,243896.736777,112124.128531,89596.982075,84807.943978,68842.056727
1,338656.012618,309862.716840,253225.594554,229272.348211,74119.107786,52134.928619,73198.336020,131310.929113
2,16441.988966,150165.050075,103156.233155,93242.100806,52134.928619,52134.928619,52134.928619,52134.928619


#### 4. Summarization

In [12]:
quantity_lion = median_summ(quantity_mat_fill, topN_rep=3, rep_abundance='sum')
np.log2(quantity_lion)

array([19.59572875, 19.57770256, 19.28649661, 19.13673503, 17.95184285,
       17.62826812, 17.54901739, 18.29800377])